# Validação territorial — população IBGE municipal e Distrito Federal

**Objetivo:** validar estimativas municipais e do Distrito Federal para 2024 e 2025, conferir chaves e conciliação interna com as UFs e descrever os maiores valores. A base pode compor denominadores anuais, mas não há numeradores SIA/SIH municipais validados para calcular taxas.


## Fonte

[IBGE — estimativas da população](https://www.ibge.gov.br/estatisticas/sociais/populacao/9103-estimativas-de-populacao.html), com referência em 1º de julho de cada ano. Os CSVs locais preservam o arquivo e a URL de origem por registro.


## 1. Leitura e consistência dos registros territoriais

In [1]:
from pathlib import Path
import csv

BASES=[Path("../Dados/02_derivados_preliminares/IBGE"),Path("../Dados"),Path("estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final/Dados"),Path("Dados")]
def ler(nome):
    p=next((b/nome for b in BASES if (b/nome).exists()),BASES[0]/nome)
    if not p.exists(): raise FileNotFoundError(p)
    with p.open(encoding="utf-8-sig",newline="") as f: return list(csv.DictReader(f,delimiter=";"))
municipios=ler("populacao_ibge_municipio_2024_2025.csv")
ufs=ler("populacao_ibge_uf_2024_2025.csv")
assert municipios and len(municipios)==len({(r["ano"],r["cod_municipio_ibge"]) for r in municipios}), "Registros município/ano duplicados ou vazios."
assert set(r["ano"] for r in municipios)=={"2024","2025"}
for r in municipios:
    r["populacao_estimada"]=int(r["populacao_estimada"])
    assert len(r["cod_municipio_ibge"])==7
    assert r["populacao_estimada"]>0
print("Registros municipais:",len(municipios))
print("Unidades municipais/DF em 2024:",len({r["cod_municipio_ibge"] for r in municipios if r["ano"]=="2024"}))
print("Unidades municipais/DF em 2025:",len({r["cod_municipio_ibge"] for r in municipios if r["ano"]=="2025"}))


Registros municipais: 11142
Unidades municipais/DF em 2024: 5571
Unidades municipais/DF em 2025: 5571


## 2. Reconciliar unidades territoriais e UFs

In [2]:
popuf={(r["ano"],r["cod_uf"]):int(r["populacao_estimada"]) for r in ufs}
por_uf={}
for r in municipios:
    chave=(r["ano"],r["cod_uf"])
    por_uf[chave]=por_uf.get(chave,0)+r["populacao_estimada"]
assert set(por_uf)==set(popuf), "Conjunto de UFs/anos difere entre arquivo municipal e arquivo UF."
divergencias=[(ano,cod,por_uf[(ano,cod)],popuf[(ano,cod)]) for ano,cod in popuf if por_uf[(ano,cod)]!=popuf[(ano,cod)]]
assert not divergencias, f"Somas municipais não conciliam com UF: {divergencias[:10]}"
for ano in ("2024","2025"):
    print(ano,"total nacional (soma das UFs):",sum(v for (a,_),v in popuf.items() if a==ano))
print("OK: a soma das unidades municipais/DF reconcilia com cada UF nos dois anos.")


2024 total nacional (soma das UFs): 212583750
2025 total nacional (soma das UFs): 213421037
OK: a soma das unidades municipais/DF reconcilia com cada UF nos dois anos.


## 3. Maiores unidades territoriais e variações estimadas

In [3]:
por_codigo={}
for r in municipios: por_codigo.setdefault(r["cod_municipio_ibge"],{})[r["ano"]]=r
assert all(set(v)=={"2024","2025"} for v in por_codigo.values()), "Município sem um dos anos."
print("10 unidades municipais/DF mais populosas em 2025:")
for r in sorted((v["2025"] for v in por_codigo.values()),key=lambda x:x["populacao_estimada"],reverse=True)[:10]:
    print(f"{r['municipio']} / {r['sigla_uf']}: {r['populacao_estimada']:,}".replace(",","."))

variacao=[]
for cod,v in por_codigo.items():
    a,b=v["2024"]["populacao_estimada"],v["2025"]["populacao_estimada"]
    variacao.append((v["2025"]["municipio"],v["2025"]["sigla_uf"],100*(b-a)/a,a,b))
print("\nMaiores aumentos percentuais estimados (2024–2025):")
for nome,uf,pct,a,b in sorted(variacao,key=lambda x:x[2],reverse=True)[:10]:
    print(f"{nome}/{uf}: {pct:.1f}% ({a:,} → {b:,})".replace(",","."))


10 unidades municipais/DF mais populosas em 2025:
São Paulo / SP: 11.904.961
Rio de Janeiro / RJ: 6.730.729
Brasília / DF: 2.996.899
Fortaleza / CE: 2.578.483
Salvador / BA: 2.564.204
Belo Horizonte / MG: 2.415.872
Manaus / AM: 2.303.732
Curitiba / PR: 1.830.795
Recife / PE: 1.588.376
Goiânia / GO: 1.503.256

Maiores aumentos percentuais estimados (2024–2025):
São Cristóvão/SE: 10.3% (100.360 → 110.737)
Sambaíba/MA: 9.3% (5.701 → 6.234)
São José do Divino/PI: 8.4% (4.906 → 5.317)
Pacaraima/RR: 4.6% (22.104 → 23.112)
Itapoá/SC: 4.3% (34.546 → 36.033)
Querência/MT: 4.3% (29.820 → 31.100)
Barra Velha/SC: 4.2% (50.730 → 52.860)
Tupanatinga/PE: 4.2% (27.009 → 28.141)
Abadia de Goiás/GO: 4.2% (21.165 → 22.052)
Passo de Torres/SC: 4.0% (14.284 → 14.859)


### Limites para uso analítico

- Cada ano contém 5.571 unidades: 5.570 municípios e o Distrito Federal.
- A conciliação é interna entre as tabelas municipal e estadual; não substitui a conferência com publicação oficial do IBGE.
- A população pode ser denominador apenas para numeradores compatíveis em geografia e período.
- Variações entre estimativas não medem, isoladamente, mudança demográfica observada; podem refletir revisão metodológica. Examine valores absolutos e a edição do IBGE.
- Não inferir acesso, necessidade clínica ou desempenho assistencial a partir da população isolada.
